# Sistema Experto: Detección de correos de phishing

1. **Base de conocimientos**: reglas SI … ENTONCES … dadas por un experto en ciberseguridad.
2. **Obtención de hechos**: el usuario describe el correo y sus respuestas se convierten en hechos.
3. **Motor de inferencia**: compara los hechos con las condiciones de cada regla.
4. **Resultados + módulo de explicación**: diagnóstico, recomendación, certeza y *por qué*.
5. **Programa principal**.

In [7]:
# ============================================================
# SISTEMA EXPERTO
# Detección de correos de phishing
# ============================================================

# ------------------------------------------------------------
# 1. BASE DE CONOCIMIENTOS
# ------------------------------------------------------------
# Las reglas representan conocimiento proporcionado por
# un experto en ciberseguridad. Cada regla tiene condiciones,
# una conclusión, una recomendación y un factor de certeza.
#
# Formato conceptual:
#
# SI condición_1 Y condición_2
# ENTONCES conclusión   (certeza)
# ------------------------------------------------------------

reglas = [

    {
        "nombre": "Regla 1 - Suplantación con robo de datos",
        "condiciones": ["remitente_sospechoso", "pide_datos"],
        "conclusion": "Phishing muy probable (intento de robo de credenciales)",
        "recomendacion": "No responder ni ingresar datos. Reportar el correo como phishing y eliminarlo.",
        "certeza": 0.95
    },

    {
        "nombre": "Regla 2 - Presión y enlace",
        "condiciones": ["urgencia", "enlace_sospechoso"],
        "conclusion": "Phishing probable (ingeniería social por urgencia)",
        "recomendacion": "No hacer clic. Ingresar al servicio escribiendo la dirección oficial en el navegador.",
        "certeza": 0.85
    },

    {
        "nombre": "Regla 3 - Adjunto peligroso",
        "condiciones": ["adjunto_peligroso"],
        "conclusion": "Riesgo de malware en el archivo adjunto",
        "recomendacion": "No abrir el adjunto. Analizarlo con un antivirus o consultar al área de TI.",
        "certeza": 0.90
    },

    {
        "nombre": "Regla 4 - Premio inesperado",
        "condiciones": ["premio_o_oferta", "remitente_desconocido"],
        "conclusion": "Estafa tipo 'ganaste un premio'",
        "recomendacion": "Ignorar el correo. Nadie regala premios por concursos en los que no participaste.",
        "certeza": 0.85
    },

    {
        "nombre": "Regla 5 - Correo mal redactado",
        "condiciones": ["errores_ortograficos", "saludo_generico"],
        "conclusion": "Correo masivo de baja confianza",
        "recomendacion": "Verificar la autenticidad antes de realizar cualquier acción.",
        "certeza": 0.60
    },

    {
        "nombre": "Regla 6 - Muchos enlaces",
        "condiciones": ["muchos_enlaces", "remitente_desconocido"],
        "conclusion": "Spam publicitario",
        "recomendacion": "Marcar como spam y, si aplica, cancelar la suscripción desde el sitio oficial.",
        "certeza": 0.65
    },

    {
        "nombre": "Regla 7 - Petición de dinero",
        "condiciones": ["pide_dinero", "urgencia"],
        "conclusion": "Fraude financiero (posible suplantación de un conocido o jefe)",
        "recomendacion": "Confirmar por otro medio (llamada o en persona) antes de transferir dinero.",
        "certeza": 0.90
    },

    {
        "nombre": "Regla 8 - Dominio falso",
        "condiciones": ["remitente_sospechoso"],
        "conclusion": "El remitente podría estar suplantando a una entidad",
        "recomendacion": "Comparar la dirección del remitente con la dirección oficial de la entidad.",
        "certeza": 0.70
    },

    {
        "nombre": "Regla 9 - Correo confiable",
        "condiciones": ["remitente_conocido", "sin_senales"],
        "conclusion": "El correo parece legítimo",
        "recomendacion": "Puede leerse con normalidad, manteniendo la precaución habitual.",
        "certeza": 0.90
    }
]

print(f"Base de conocimientos cargada: {len(reglas)} reglas.")

Base de conocimientos cargada: 9 reglas.


In [8]:
# ------------------------------------------------------------
# 2. OBTENER LOS HECHOS DEL CORREO
# ------------------------------------------------------------
# Los hechos representan la información conocida sobre el correo.

def preguntar_si_no(pregunta):
    """Repite la pregunta hasta recibir 's' o 'n'."""
    while True:
        respuesta = input(pregunta + " (s/n): ").strip().lower()
        if respuesta in ("s", "n"):
            return respuesta == "s"
        print("  Respuesta no válida. Escriba 's' o 'n'.")


def preguntar_numero(pregunta, minimo, maximo):
    """Repite la pregunta hasta recibir un número dentro del rango."""
    while True:
        try:
            valor = int(input(pregunta + f" ({minimo}-{maximo}): "))
            if minimo <= valor <= maximo:
                return valor
        except ValueError:
            pass
        print(f"  Ingrese un número entero entre {minimo} y {maximo}.")


def obtener_hechos():

    print("\n======================================")
    print("============ INTERFAZ ================")
    print("  SISTEMA EXPERTO - DETECTOR PHISHING")
    print("======================================")

    hechos = set()

    # --------------------------------------------------------
    # Parte A: el remitente
    # --------------------------------------------------------
    if preguntar_si_no("¿Conoces al remitente o esperabas este correo?"):
        hechos.add("remitente_conocido")
    else:
        hechos.add("remitente_desconocido")

    if preguntar_si_no("¿La dirección del remitente se ve rara? (ej. banco-seguro123@gmail.com)"):
        hechos.add("remitente_sospechoso")

    # --------------------------------------------------------
    # Parte B: el contenido
    # --------------------------------------------------------
    if preguntar_si_no("¿Pide contraseñas, códigos o datos personales/bancarios?"):
        hechos.add("pide_datos")

    if preguntar_si_no("¿Pide transferir dinero o comprar tarjetas de regalo?"):
        hechos.add("pide_dinero")

    if preguntar_si_no("¿Transmite urgencia o amenaza? (ej. 'su cuenta será bloqueada hoy')"):
        hechos.add("urgencia")

    if preguntar_si_no("¿Ofrece premios, sorteos u ofertas increíbles?"):
        hechos.add("premio_o_oferta")

    if preguntar_si_no("¿Tiene errores ortográficos o de redacción?"):
        hechos.add("errores_ortograficos")

    if preguntar_si_no("¿Usa un saludo genérico? (ej. 'Estimado cliente')"):
        hechos.add("saludo_generico")

    # --------------------------------------------------------
    # Parte C: enlaces y adjuntos
    # --------------------------------------------------------
    if preguntar_si_no("Al pasar el mouse sobre un enlace, ¿la dirección no coincide o está acortada?"):
        hechos.add("enlace_sospechoso")

    if preguntar_si_no("¿Trae un adjunto .exe, .zip, .js o un documento que pide 'habilitar macros'?"):
        hechos.add("adjunto_peligroso")

    # Dato numérico -> lo convertimos en HECHO
    enlaces = preguntar_numero("¿Cuántos enlaces contiene el correo?", 0, 100)
    if enlaces >= 5:
        hechos.add("muchos_enlaces")

    # --------------------------------------------------------
    # Si no hay ninguna señal de alerta, lo registramos como hecho.
    # --------------------------------------------------------
    senales = {"remitente_sospechoso", "pide_datos", "pide_dinero", "urgencia",
               "premio_o_oferta", "errores_ortograficos", "saludo_generico",
               "enlace_sospechoso", "adjunto_peligroso", "muchos_enlaces"}
    if not (hechos & senales):
        hechos.add("sin_senales")

    return hechos

In [9]:
# ------------------------------------------------------------
# 3. MOTOR DE INFERENCIA
# ------------------------------------------------------------

def motor_inferencia(hechos):
    """
    Compara los hechos conocidos con las condiciones
    de cada regla (encadenamiento hacia adelante).

    Si todas las condiciones de una regla están presentes,
    la regla se activa y genera una conclusión.
    """

    resultados = []

    print("\n======================================")
    print("        MOTOR DE INFERENCIA")
    print("======================================")

    print("\nHechos detectados:")

    for hecho in sorted(hechos):
        print("  •", hecho)

    print("\nAnalizando reglas...\n")

    for regla in reglas:

        # ----------------------------------------------------
        # Verificamos si TODAS las condiciones de la regla
        # están presentes en los hechos.
        # ----------------------------------------------------

        if all(condicion in hechos
               for condicion in regla["condiciones"]):

            print("✓ Regla activada:", regla["nombre"])
            resultados.append(regla)

        else:
            print("✗ Regla descartada:", regla["nombre"])

    # Primero los diagnósticos con mayor certeza.
    resultados.sort(key=lambda r: r["certeza"], reverse=True)

    return resultados

In [10]:
# ------------------------------------------------------------
# 4. MOSTRAR CONCLUSIONES + MÓDULO DE EXPLICACIÓN
# ------------------------------------------------------------

def nivel_riesgo(resultados):
    """Resume el riesgo según la regla de mayor certeza."""
    if not resultados:
        return "INDETERMINADO"
    if resultados[0]["nombre"].startswith("Regla 9"):
        return "BAJO"
    if resultados[0]["certeza"] >= 0.85:
        return "ALTO"
    return "MEDIO"


def mostrar_resultados(resultados):

    print("\n======================================")
    print("          RESULTADO DEL SISTEMA")
    print("======================================")

    if not resultados:

        print("\nNo se encontró un patrón específico.")
        print("Ante la duda, verifique el correo con el área de TI.")

        return

    print("\nNIVEL DE RIESGO:", nivel_riesgo(resultados))

    for i, resultado in enumerate(resultados, start=1):

        print(f"\n[{i}] Diagnóstico:")
        print("→", resultado["conclusion"])

        print("   Certeza:", f"{resultado['certeza'] * 100:.0f}%")

        print("   Recomendación:")
        print("   →", resultado["recomendacion"])

        # Módulo de explicación: ¿por qué llegó a esta conclusión?
        print("   ¿Por qué?", resultado["nombre"],
              "se activó porque se cumplieron:",
              " Y ".join(resultado["condiciones"]))

### Prueba rápida (sin teclado)

Casos predefinidos para verificar las reglas antes de usar la interfaz.

In [11]:
# ------------------------------------------------------------
# 5. PROGRAMA PRINCIPAL
# ------------------------------------------------------------

def main():

    # Obtener información del correo
    hechos = obtener_hechos()

    # Ejecutar el razonamiento
    resultados = motor_inferencia(hechos)

    # Mostrar las conclusiones
    mostrar_resultados(resultados)


main()


============ INTERFAZ ================
  SISTEMA EXPERTO - DETECTOR PHISHING
¿Conoces al remitente o esperabas este correo? (s/n): s
¿La dirección del remitente se ve rara? (ej. banco-seguro123@gmail.com) (s/n): s
¿Pide contraseñas, códigos o datos personales/bancarios? (s/n): s
¿Pide transferir dinero o comprar tarjetas de regalo? (s/n): n
¿Transmite urgencia o amenaza? (ej. 'su cuenta será bloqueada hoy') (s/n): n
¿Ofrece premios, sorteos u ofertas increíbles? (s/n): s
¿Tiene errores ortográficos o de redacción? (s/n): s
¿Usa un saludo genérico? (ej. 'Estimado cliente') (s/n): s
Al pasar el mouse sobre un enlace, ¿la dirección no coincide o está acortada? (s/n): s
¿Trae un adjunto .exe, .zip, .js o un documento que pide 'habilitar macros'? (s/n): s
¿Cuántos enlaces contiene el correo? (0-100): 2

        MOTOR DE INFERENCIA

Hechos detectados:
  • adjunto_peligroso
  • enlace_sospechoso
  • errores_ortograficos
  • pide_datos
  • premio_o_oferta
  • remitente_conocido
  • remitente_